# Day 27 — Animation
### #30DayChartChallenge | April 2026

**The Energy Race: 224 years of shifting fuel ranks (1800–2024).**

**Tool:** `gganimate` · bar chart race with reordering (`transition_states`)

**Data:** [Our World in Data — Global primary energy consumption by source](https://ourworldindata.org/grapher/global-energy-substitution) · Energy Institute Statistical Review of World Energy 2025 + Smil (2017), substitution method, TWh  
**Author:** Sharfudeen Yasar Arafath

In [34]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(showtext)
library(sysfonts)
library(scales)
library(gganimate)
library(gifski)

In [35]:
font_add_google("Outfit", "outfit")
font_add_google("Roboto Condensed", "roboto_condensed")
font_add_google("JetBrains Mono", "jetbrains")
showtext_auto()
showtext_opts(dpi = 150)

options(repr.plot.width = 13, repr.plot.height = 8, repr.plot.res = 150)

In [ ]:
bg_col   <- "#0a0e17"
text_col <- "#e0e6f0"
grid_col <- "#1a2030"

# 10 fuels in OWID's global-energy-substitution dataset.
# Earth tones for biomass + fossils, vivid blues/greens/yellow for low-carbon.
source_colours <- c(
  "Traditional biomass" = "#7a5c3a",
  "Coal"                = "#3d3d3d",
  "Oil"                 = "#8a3324",
  "Natural gas"         = "#c87533",
  "Nuclear"             = "#a06ec5",
  "Hydropower"          = "#118ab2",
  "Wind"                = "#06d6a0",
  "Solar"               = "#ffd166",
  "Modern biofuels"     = "#83c5be",
  "Other renewables"    = "#e0e6f0"
)

In [ ]:
raw <- read.csv("../../data/day_27/owid_energy_official.csv",
                stringsAsFactors = FALSE, check.names = FALSE)

long <- raw %>%
  filter(Entity == "World") %>%
  select(year = Year,
         `Traditional biomass`, Coal, Oil, `Natural gas`, Nuclear,
         Hydropower, Wind, Solar, `Modern biofuels`, `Other renewables`) %>%
  pivot_longer(-year, names_to = "source", values_to = "twh") %>%
  mutate(twh = ifelse(is.na(twh), 0, twh))

# 2024 final ranks — used (a) as a stable tiebreaker when many fuels are 0,
# and (b) to pre-arrange all 10 labels in the intro frame.
final_rank <- long %>%
  filter(year == 2024) %>%
  mutate(final_rank = rank(-twh, ties.method = "first")) %>%
  select(source, final_rank)

# Frame schedule: an "intro" frame at 1799 (all zeros, all 10 labels visible),
# then decade up to 1960, 5y to 1995, yearly 2000-2024.
frame_years <- c(1799, sort(unique(c(
  seq(1800, 1960, by = 10),
  seq(1965, 1995, by = 5),
  seq(2000, 2024, by = 1)
))))

# Build intro rows (all sources, twh = 0) for the 1799 hold.
intro <- final_rank %>%
  mutate(year = 1799, twh = 0) %>%
  select(year, source, twh)

# Race data: rank by twh, break ties by 2024 size so labels stay in a stable
# top-to-bottom order even when many bars are at zero.
race <- bind_rows(intro, long) %>%
  filter(year %in% frame_years) %>%
  left_join(final_rank, by = "source") %>%
  group_by(year) %>%
  arrange(desc(twh), final_rank, .by_group = TRUE) %>%
  mutate(rank = row_number()) %>%
  ungroup() %>%
  mutate(source_label = as.character(source))

cat("Frames:", length(frame_years), "| Years:", range(frame_years), "\n")
cat("Rows:", nrow(race), "| Sources/frame:", n_distinct(race$source), "\n")
cat("Intro (1799) ranks:\n")
print(race %>% filter(year == 1799) %>% select(rank, source) %>% arrange(rank))

In [ ]:
chart_theme <- theme_minimal(base_family = "outfit", base_size = 14) +
  theme(
    plot.background    = element_rect(fill = bg_col, colour = NA),
    panel.background   = element_rect(fill = bg_col, colour = NA),
    panel.grid.major.x = element_line(colour = grid_col, linewidth = 0.3),
    panel.grid.minor   = element_blank(),
    panel.grid.major.y = element_blank(),
    plot.title    = element_text(family = "outfit", face = "bold",
                                size = 28, colour = text_col,
                                margin = margin(b = 4)),
    plot.subtitle = element_text(family = "roboto_condensed",
                                size = 13, colour = alpha(text_col, 0.7),
                                margin = margin(b = 18)),
    plot.caption  = element_text(family = "jetbrains", size = 8,
                                colour = alpha(text_col, 0.5),
                                hjust = 1, lineheight = 1.5,
                                margin = margin(t = 28)),
    axis.text.x   = element_text(family = "jetbrains", size = 10,
                                colour = alpha(text_col, 0.6),
                                margin = margin(t = 6)),
    axis.text.y   = element_blank(),
    axis.title    = element_blank(),
    axis.ticks    = element_blank(),
    legend.position = "none",
    plot.margin   = margin(20, 30, 30, 200)
  )

chart_caption <- "#30DayChartChallenge · Day 27 · Animation (gganimate)\nData: Our World in Data · EI Statistical Review of World Energy 2025 + Smil 2017\nSubstitution method · 10 fuel categories · Author: Sharfudeen Yasar Arafath"

x_max <- max(race$twh) * 1.05

In [ ]:
# Bar chart race: bars reorder by rank as the year advances.
# source_label is constant per source group, so geom_text label tweens cleanly.
p_anim <- ggplot(race, aes(x = -rank, y = twh, fill = source, group = source)) +
  geom_col(width = 0.78, alpha = 0.95) +
  geom_text(aes(label = source_label, y = 0),
            family = "outfit", fontface = "bold",
            colour = text_col, hjust = 1.05, size = 5) +
  coord_flip(clip = "off") +
  scale_y_continuous(labels = label_comma(),
                     limits = c(0, x_max),
                     expand = expansion(mult = c(0, 0.02))) +
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) +
  scale_fill_manual(values = source_colours) +
  labs(
    title    = "The Energy Race · {closest_state}",
    subtitle = "Global energy consumption by fuel · ranked each year, 1800–2024 (TWh)",
    caption  = chart_caption
  ) +
  chart_theme +
  transition_states(year, transition_length = 4, state_length = 1) +
  ease_aes("cubic-in-out")

# start_pause holds the 1800 frame; end_pause holds the 2024 frame
anim <- suppressWarnings(animate(
  p_anim,
  nframes     = 400,
  fps         = 22,
  width       = 1700,
  height      = 1000,
  res         = 150,
  renderer    = gifski_renderer(),
  start_pause = 60,
  end_pause   = 120
))

anim_save("../../chart/day_27_animation.gif", animation = anim)
cat("✅ GIF saved\n")

In [ ]:
final_year <- max(race$year)
race_final <- race %>% filter(year == final_year)

p_static <- ggplot(race_final, aes(x = -rank, y = twh, fill = source)) +
  geom_col(width = 0.78, alpha = 0.95) +
  geom_text(aes(label = source_label, y = 0),
            family = "outfit", fontface = "bold",
            colour = text_col, hjust = 1.05, size = 5) +
  geom_text(aes(label = scales::comma(round(twh))),
            family = "jetbrains", colour = text_col,
            hjust = -0.15, size = 4.2) +
  coord_flip(clip = "off") +
  scale_y_continuous(labels = label_comma(),
                     limits = c(0, x_max),
                     expand = expansion(mult = c(0, 0.05))) +
  scale_x_continuous(expand = expansion(mult = c(0.05, 0.05))) +
  scale_fill_manual(values = source_colours) +
  labs(
    title    = paste0("The Energy Race · ", final_year),
    subtitle = "Global energy consumption by fuel · ranked, 1800–2024 (TWh)",
    caption  = chart_caption
  ) +
  chart_theme

showtext_opts(dpi = 300)
ggsave("../../chart/day_27_animation.png", plot = p_static,
       width = 14, height = 8.5, dpi = 300, bg = bg_col)
cat("✅ PNG saved\n")